# 31 — Exploration: missingness, imbalance, and the 7-point rule

**Reads:** `silver.knee_episode`, `gold.knee_features` · **Segment 3 of the session**

Ported from `01_all_dataset_exploration` and `02_knee_data_exploration`, trimmed to the
four things that have to survive contact with a Level 300 audience:

1. **Why 7 points?** The target definition is a clinical decision, not a modelling one, and
   it deserves to be defended before any model appears.
2. **Where is the data missing, and is it missing at random?** This is what justifies MICE
   over dropping rows or mean-filling.
3. **How imbalanced is the poor-outcome class?** This is what justifies precision–recall
   over accuracy.
4. **How many patients have no outcome at all?** — the finding that changed the pipeline.

Everything here is a read. Nothing in this notebook writes.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F

sns.set_style("whitegrid")
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "font.size": 10,
})

SILVER = "silver.knee_episode"
GOLD   = "gold.knee_features"
MCID   = 7

silver = spark.table(SILVER)
gold   = spark.table(GOLD)
print(f"silver: {silver.count():,} episodes")
print(f"gold:   {gold.count():,} episodes")

## 2. Why 7 points? The minimal clinically important difference

The Oxford Knee Score runs 0–48. A patient whose score rises by 2 points has technically
improved and will tell you nothing has changed. The **MCID** is the smallest change a
patient actually notices — for the OKS, around 7 points.

That is the entire basis of the target. Everything the model does is anchored to a number
that came from patients, not from a validation curve. Say that out loud before showing a
single metric.

The distribution below is the argument: it is wide, it straddles zero, and a meaningful
tail of patients get little or nothing from a technically successful operation.

In [ ]:
delta = (gold.filter("has_label = true")
             .select("oks_delta", "oks_t0_score", "outcome_label")
             .toPandas())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
ax.hist(delta["oks_delta"], bins=60, color="#4C78A8", edgecolor="white", linewidth=.4)
ax.axvline(MCID, color="#C44E52", lw=2, ls="--", label=f"MCID = {MCID} points")
ax.axvline(0, color="#555", lw=1)
ax.set_xlabel("OKS improvement (post-op minus pre-op)")
ax.set_ylabel("Patients")
ax.set_title("Not everyone improves meaningfully")
ax.legend()

below = (delta["oks_delta"] <= MCID).mean()
ax.annotate(f"{below:.0%} at or below MCID",
            xy=(MCID, ax.get_ylim()[1] * .72),
            xytext=(MCID - 34, ax.get_ylim()[1] * .88),
            arrowprops=dict(arrowstyle="->", color="#C44E52"), color="#C44E52", fontweight="bold")

# Pre-op score vs. gain: the single strongest signal in the whole dataset.
ax = axes[1]
binned = delta.groupby(pd.cut(delta["oks_t0_score"], bins=range(0, 50, 4)), observed=True).agg(
    mean_gain=("oks_delta", "mean"), n=("oks_delta", "size")).reset_index()
centres = [iv.mid for iv in binned["oks_t0_score"]]
ax.plot(centres, binned["mean_gain"], "o-", color="#4C78A8")
ax.axhline(MCID, color="#C44E52", lw=2, ls="--")
ax.set_xlabel("Pre-operative OKS")
ax.set_ylabel("Mean improvement")
ax.set_title("The worse you start, the more you gain")

plt.tight_layout(); plt.show()

print(f"Mean improvement:   {delta['oks_delta'].mean():.1f} points")
print(f"Median improvement: {delta['oks_delta'].median():.1f} points")
print(f"At or below MCID:   {below:.1%}  <- these are the patients we are trying to find")

### The counter-intuitive bit worth saying on stage

The right-hand panel shows mean gain *falling* as pre-op score rises — patients who start
worse improve more. That is partly real (more room to improve) and partly regression to the
mean.

It also means the naive clinical heuristic — *"the worse they are, the more they need
surgery"* — is roughly the opposite of what predicts a poor outcome. The model is not
automating an existing rule of thumb. It is finding patients the rule of thumb misses, and
that is the reason to build it at all.

## 3. Knee vs. hip — is the problem knee-specific?

The original exploration compared both procedures before committing to knee. Worth keeping:
it shows the cohort choice was made, not assumed. Hip replacement produces larger and more
reliable gains, which is precisely why knee is the more interesting prediction problem.

Reads the legacy interim parquet, since bronze only ingests knee.

In [ ]:
try:
    hip = (spark.read.parquet("Files/data/interim/hip-provider.parquet")
                .selectExpr("ohs_t0_score AS t0", "ohs_t1_score AS t1")
                .filter("t0 IS NOT NULL AND t1 IS NOT NULL")
                .withColumn("gain", F.col("t1") - F.col("t0"))
                .select("gain").toPandas())
    hip["procedure"] = "Hip (OHS)"

    knee = delta[["oks_delta"]].rename(columns={"oks_delta": "gain"}).copy()
    knee["procedure"] = "Knee (OKS)"
    both = pd.concat([knee, hip], ignore_index=True)

    fig, ax = plt.subplots(figsize=(8, 4))
    for proc, colour in [("Knee (OKS)", "#4C78A8"), ("Hip (OHS)", "#72B7B2")]:
        s = both.loc[both["procedure"] == proc, "gain"]
        ax.hist(s, bins=60, alpha=.6, label=f"{proc}  (median {s.median():.0f})",
                color=colour, density=True)
    ax.axvline(MCID, color="#C44E52", lw=2, ls="--", label=f"MCID = {MCID}")
    ax.set_xlabel("Score improvement"); ax.set_ylabel("Density")
    ax.set_title("Hip replacement delivers more, more reliably")
    ax.legend(); plt.tight_layout(); plt.show()

    print(both.groupby("procedure")["gain"].agg(["count", "mean", "median"]).round(1).to_string())
    for proc in both["procedure"].unique():
        s = both.loc[both["procedure"] == proc, "gain"]
        print(f"{proc}: {(s <= MCID).mean():.1%} at or below MCID")
except Exception as e:
    print("Hip comparison skipped (legacy parquet not found):", e)

## 4. Class imbalance — why accuracy is the wrong metric

Roughly one patient in five has a poor outcome. A model that predicts "good outcome" for
everybody scores about 80% accuracy and finds nobody who needs pre-operative optimisation.

The bar below is the honest baseline that every metric in notebook 40 is measured against.

In [ ]:
counts = (gold.filter("has_label = true").groupBy("outcome_label").count()
              .orderBy("outcome_label").toPandas())
counts["label"] = counts["outcome_label"].map({0: "Meaningful improvement", 1: "Poor outcome"})
total = counts["count"].sum()
prevalence = counts.loc[counts["outcome_label"] == 1, "count"].iloc[0] / total

fig, ax = plt.subplots(figsize=(7.5, 2.6))
left = 0
for _, r in counts.iterrows():
    colour = "#C44E52" if r["outcome_label"] == 1 else "#9EC1D9"
    ax.barh(0, r["count"], left=left, color=colour, edgecolor="white")
    ax.text(left + r["count"] / 2, 0, f"{r['label']}\n{r['count']:,}  ({r['count']/total:.0%})",
            ha="center", va="center", fontsize=10,
            color="white" if r["outcome_label"] == 1 else "#22303A", fontweight="bold")
    left += r["count"]
ax.set_yticks([]); ax.set_xlim(0, total); ax.set_xlabel("Labelled episodes")
ax.set_title("The class we care about is the minority one")
for s in ax.spines.values():
    s.set_visible(False)
plt.tight_layout(); plt.show()

print(f"Poor-outcome prevalence: {prevalence:.1%}")
print(f"Accuracy of a 'nobody has a poor outcome' model: {1 - prevalence:.1%}")
print(f"Poor outcomes it would find: 0 of {counts.loc[counts.outcome_label==1,'count'].iloc[0]:,}")
print(f"\nNo-skill average precision (the PR-curve baseline): {prevalence:.3f}")

## 5. The patients with no outcome at all

**This one changed the pipeline, and it is worth showing.**

Not every patient returns the six-month questionnaire. Those episodes have a pre-operative
score and no post-operative score, so `oks_delta` is null and no label can be computed.

The original notebook computed the target as:

```python
df['oks_target_variable'] = np.where(df['oks_delta'] > 7, 0, 1)
```

`np.nan > 7` evaluates to `False`, so every non-responder fell into the `else` branch and
was labelled **1 — poor outcome**. Silent, and entirely plausible-looking downstream.

The cell below sizes it against the current gold table. The pipeline now marks these rows
`has_label = false` and notebook 40 excludes them, so the model is trained only on patients
whose outcome is actually known.

In [ ]:
tot        = gold.count()
labelled   = gold.filter("has_label = true").count()
unlabelled = tot - labelled
positives  = gold.filter("outcome_label = 1").count()

print(f"Episodes in gold:            {tot:,}")
print(f"  with a known outcome:      {labelled:,}")
print(f"  no follow-up questionnaire:{unlabelled:,}  ({unlabelled/tot:.1%})")
print()
print("If those had been swept into the positive class, as in the original notebook:")
print(f"  true poor outcomes:        {positives:,}")
print(f"  non-responders added:      {unlabelled:,}")
contaminated = positives + unlabelled
print(f"  contaminated positives:    {contaminated:,}")
print(f"  -> {unlabelled/contaminated:.1%} of the minority class would be "
      f"'we do not know' mislabelled as 'poor outcome'")

# Are non-responders different from responders? If they are, dropping them is not neutral
# either -- it is informative missingness, and the model card should say so.
comp = (gold.groupBy("has_label")
            .agg(F.count("*").alias("n"),
                 F.round(F.avg("oks_t0_score"), 1).alias("mean_preop_oks"),
                 F.round(F.avg("comorbidity_count"), 2).alias("mean_comorbidities"))
            .toPandas())
print()
print(comp.to_string(index=False))
print("\nIf these rows differ materially, note it as a limitation: the training population "
      "is patients who responded, which is not quite the population the model will score.")

## 6. Missingness — and why it is handled two different ways

Two patterns, two mechanisms, two treatments. Conflating them is the mistake.

### `age_band` and `gender` — suppressed by NHS, not missed by the patient

From the data pack's own documentation (`CSV Footnotes 1819 Finalised.pdf`, *Patient-level
CSV files → Low numbers*):

> "To protect patient confidentiality, records have been excluded where there would otherwise
> have been fewer than 6 records present for any provider. Where there are between 1 and 5
> records for a particular combination of **ageband, sex, provider and procedure**, the
> ageband and sex have been suppressed and replaced with an asterisk (\*). Where it would have
> remained possible to identify suppressed cells by derivation from the total, additional
> cells have been suppressed."

Three consequences that change what we are allowed to do:

1. **They are suppressed as a pair**, by one rule — which is why they are missing on exactly
   the same rows.
2. **The trigger is a small (ageband, sex, provider, procedure) cell**, so a starred row is
   one where that patient's demographic combination is *rare at that provider* — precisely
   what made it identifying.
3. **Some suppression is complementary** — starred not because its own cell was small, but
   because leaving it visible would let you derive a suppressed cell from the totals.

So imputing these would mean training a model to reconstruct the value NHS hid to protect
that patient's confidentiality, then serving clinical decisions on the reconstruction. That
is a question for a data protection officer, not a modelling choice. **We do not impute
them** — gold gives them an explicit `not_recorded` level instead.

Sex is also a protected characteristic. Imputing it would make the bias audit in
`42_threshold_analysis` §7 partly an audit of our own imputer.

### The questionnaire items — genuine patient non-response

`t0_mobility`, `t0_anxiety` and friends are missing individually, at low rates, in the
pattern you would expect from people skipping questions. Not protected, not suppressed,
plausibly missing-at-random conditional on the other answers. **These we impute with MICE**,
which models each missing column from the others rather than reaching for a population mean.

The line: *impute questionnaire non-response, never impute a protected characteristic.*

In [ ]:
IMPUTED_COLS = [
    "age_band", "gender", "t0_symptom_period", "t0_living_arrangements", "t0_assisted",
    "t0_assisted_by", "t0_previous_surgery", "t0_disability",
    "t0_mobility", "t0_self_care", "t0_activity", "t0_discomfort", "t0_anxiety",
]

n = silver.count()
miss = (silver.select([
            (F.sum(F.col(c).isNull().cast("int")) / F.lit(n) * 100).alias(c)
            for c in IMPUTED_COLS])
        .toPandas().T.rename(columns={0: "pct"}).sort_values("pct"))

fig, ax = plt.subplots(figsize=(8, 4.6))
colours = ["#C44E52" if v > 2 else "#4C78A8" for v in miss["pct"]]
ax.barh(miss.index, miss["pct"], color=colours)
for i, v in enumerate(miss["pct"]):
    ax.text(v + .05, i, f"{v:.2f}%", va="center", fontsize=8.5)
ax.set_xlabel("% missing in silver")
ax.set_title("Missingness by column — red is the disclosure-control pair")
plt.tight_layout(); plt.show()

In [ ]:
# Are age_band and gender missing on exactly the same rows?
pair = silver.select(
    F.col("age_band").isNull().alias("age_missing"),
    F.col("gender").isNull().alias("gender_missing"),
).groupBy("age_missing", "gender_missing").count().toPandas()
print(pair.to_string(index=False))

both_only = pair[(pair.age_missing) & (pair.gender_missing)]["count"].sum()
age_only  = pair[(pair.age_missing) & (~pair.gender_missing)]["count"].sum()
gen_only  = pair[(~pair.age_missing) & (pair.gender_missing)]["count"].sum()
print(f"\nboth missing: {both_only:,} | age only: {age_only:,} | gender only: {gen_only:,}")
print("Near-perfect overlap => a single suppression rule, not two independent gaps.")

In [ ]:
# The documentation makes a falsifiable claim: cells of 1-5 records were suppressed.
# So among the rows that survived unsuppressed, no (ageband, sex, provider) cell should
# be smaller than 6. Test it rather than trusting it.
cell_sizes = (silver
    .filter("age_band IS NOT NULL AND gender IS NOT NULL")
    .groupBy("age_band", "gender", "provider_code")
    .agg(F.count("*").alias("cell_size"))
    .toPandas())

print("Observed (ageband, sex, provider) cell sizes, unsuppressed rows only:")
print(f"  minimum: {cell_sizes['cell_size'].min()}   <- documentation implies >= 6")
print(f"  cells with 1-5 records: {(cell_sizes['cell_size'] < 6).sum()} "
      f"of {len(cell_sizes):,}")
print()
dist = cell_sizes["cell_size"].value_counts().sort_index().head(12)
print("Distribution of the smallest cells:")
for size, n in dist.items():
    print(f"  size {size:>3}: {n:>5} cells  {'#' * min(int(n / max(dist.max(), 1) * 40), 40)}")
print()
print("A sharp floor at 6 confirms the documented rule. A few cells below it are expected -- "
      "the rule keys on (ageband, sex, provider, PROCEDURE) and this cohort is knee-only, "
      "so a cell can look small here while the full hip+knee cell cleared the threshold.")

In [ ]:
# Secondary observation: suppression correlates with provider size. This is a *proxy* --
# the actual trigger is cell size, and small providers simply have small cells -- but it is
# the operationally relevant view, because it says who gets under-represented if you drop.
by_provider = (silver
    .groupBy("provider_code")
    .agg(F.count("*").alias("episodes"),
         F.avg(F.col("age_band").isNull().cast("double")).alias("pct_suppressed"))
    .filter("episodes >= 20")
    .toPandas())

by_provider["size_band"] = pd.qcut(by_provider["episodes"], 4,
                                   labels=["smallest 25%", "2nd", "3rd", "largest 25%"])
summary = by_provider.groupby("size_band", observed=True).agg(
    providers=("provider_code", "size"),
    mean_pct_suppressed=("pct_suppressed", lambda s: round(s.mean() * 100, 2))).reset_index()
print(summary.to_string(index=False))
print("\nDropping suppressed rows would systematically under-represent small hospitals. "
      "We keep them, with an explicit 'not_recorded' level -- neither dropped nor invented.")

## 7. The sentinel trap

NHS PROMs encodes "not answered" as `9` — **except** on the twelve comorbidity flags, where
`1` means *yes* and `9` means *no*. Same code, opposite meaning, in the same file.

Read `9` as missing on a comorbidity and you invert twelve features at once. Read it as a
value everywhere else and you train a model that thinks mobility runs 1–9 on a 1–3 scale.

Silver decodes both correctly. This cell proves it, and it is a good ten seconds on stage —
it is the kind of domain detail that no amount of AutoML finds for you.

In [ ]:
print("Comorbidity flags after silver decoding (expect only 0 / 1 / null):")
for c in ["heart_disease", "diabetes", "depression"]:
    vals = silver.groupBy(c).count().orderBy(c).toPandas()
    print(f"  {c:16s} {dict(zip(vals[c].astype('object'), vals['count']))}")

print("\nEQ-5D items after decoding (expect 1 / 2 / 3 / null, never 9):")
for c in ["t0_mobility", "t0_self_care", "t0_anxiety"]:
    vals = silver.groupBy(c).count().orderBy(c).toPandas()
    print(f"  {c:16s} {dict(zip(vals[c].astype('object'), vals['count']))}")

assert silver.filter("t0_mobility = 9").count() == 0, "Sentinel 9 survived into silver"
assert silver.filter("heart_disease = 9").count() == 0, "Comorbidity 9 was not recoded to 0"
print("\nBoth decodes verified.")

## 8. Feature landscape

A quick orientation before modelling: what correlates with a poor outcome, and how strongly.
Nothing here is a causal claim — it is the map you use to sanity-check the model's
explanations later. If the EBM says something contradicts this, one of the two is wrong.

In [ ]:
NUMERIC = ["oks_t0_score", "oks_pain_subscale", "oks_function_subscale", "oks_adl_subscale",
           "comorbidity_count", "t0_symptom_period", "t0_mobility", "t0_self_care",
           "t0_activity", "t0_discomfort", "t0_anxiety", "university_hospital",
           "independent_hospital"]

pdf = gold.filter("has_label = true").select(*NUMERIC, "outcome_label").toPandas()
corr = pdf.corr(numeric_only=True)["outcome_label"].drop("outcome_label").sort_values()

fig, ax = plt.subplots(figsize=(8, 4.6))
ax.barh(corr.index, corr.values,
        color=["#C44E52" if v > 0 else "#4C78A8" for v in corr.values])
ax.axvline(0, color="#333", lw=1)
ax.set_xlabel("Correlation with poor outcome")
ax.set_title("Red increases risk, blue reduces it")
plt.tight_layout(); plt.show()

print(corr.round(3).to_string())
print("\nAll modest — no single feature separates the classes. That is why an additive "
      "model over many weak signals beats any threshold rule a clinician could apply by hand.")

## 9. What this notebook establishes

Carry these four into segment 4; they are the justification for every modelling choice that
follows.

| Finding | What it justifies |
|---|---|
| MCID = 7 points, and a fifth of patients fall below it | the target definition, and that it came from patients |
| Poor outcome is ~20% of the cohort | precision–recall, not accuracy; the PR baseline to beat |
| Demographics are suppressed by NHS disclosure control, not missed by patients | an explicit `not_recorded` level — never impute a protected characteristic |
| Questionnaire items are genuine non-response | MICE, rather than dropping rows or mean-filling |
| Non-responders have no outcome and must not be labelled | `has_label`, and an honest minority class |

---

**Next:** `40_train_register` to fit and register, or `41_model_search` if you want the
hyperparameter search that chose the parameters.